In [25]:
from pathlib import Path

ROOT = Path(".").resolve().parents[1]
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [26]:
from rich import print as rprint

In [27]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [29]:
from src.application.contracts import PipelineRequest
from src.application import ViRAGEPipeline
from src.application.settings import ViRAGESettings

tmp_path = Path("../demo_data/tmp_folder")
data_path = Path("../demo_data/Iris.csv")
settings = ViRAGESettings(artifact_root=tmp_path / "artifacts")
pipeline = ViRAGEPipeline(settings)
query="Show the sales trend over time"
request = PipelineRequest(query=query, data_path=data_path.as_posix())
result = pipeline.invoke(request)

In [30]:
rprint(result)

PipelineResult(
    run_id='3d4c2f9a76ea41ac9738e05e6f7fec47',
    query='Show the sales trend over time',
    data_path='../demo_data/Iris.csv',
    case_type=<ChartCaseType.CANONICAL: 'canonical'>,
    query_understanding=QueryUnderstandingResult(
        intent='Show the sales trend over time',
        requested_operations=['trend analysis'],
        candidate_charts=['line'],
        constraints=[],
        case_type=<ChartCaseType.CANONICAL: 'canonical'>,
        confidence=0.8
    ),
    planning=PlanningResult(
        mode=<ChartCaseType.CANONICAL: 'canonical'>,
        steps=[
            PlanningStep(
                name='profile_the_dataset_and_confirm_field',
                description='Profile the dataset and confirm field types relevant to the request.'
            ),
            PlanningStep(
                name='prepare_a_cleaned_analysis_ready_version',
                description='Prepare a cleaned analysis-ready version of the data.'
            ),
            PlanningStep(
                name='retrieve_concise_charting_guidance_for_the',
                description='Retrieve concise charting guidance for the selected chart family.'
            ),
            PlanningStep(
                name='build_the_primary_requested_chart_using',
                description='Build the primary requested chart using the leading chart family: line.'
            ),
            PlanningStep(
                name='execute_plotting_code_and_collect_numeric',
                description='Execute plotting code and collect numeric summaries from the run.'
            ),
            PlanningStep(
                name='read_chart_structure,_extract_facts_and',
                description='Read chart structure, extract facts and verify that final statements are 
evidence-backed.'
            )
        ],
        success_criteria=[
            'At least one valid canonical chart is produced, preferably among: line.',
            'Generated charts are readable and consistent with the request.',
            'Final statements reference execution metrics or chart evidence.'
        ]
    ),
    data_profile=DataProfile(
        row_count=150,
        col_count=6,
        columns=[
            DataColumnProfile(name='Id', dtype='numeric', missing_ratio=0.0, unique_count=150),
            DataColumnProfile(name='SepalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=35),
            DataColumnProfile(name='SepalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=23),
            DataColumnProfile(name='PetalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=43),
            DataColumnProfile(name='PetalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=22),
            DataColumnProfile(name='Species', dtype='categorical', missing_ratio=0.0, unique_count=3)
        ],
        likely_numeric_columns=['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm'],
        likely_categorical_columns=['Species'],
        likely_time_columns=[],
        quality_notes=["Column 'Id' looks like an identifier."]
    ),
    data_preparation=DataPreparationResult(
        output_path='../demo_data/tmp_folder/artifacts/3d4c2f9a76ea41ac9738e05e6f7fec47/cleaned_data.csv',
        operations=[],
        row_count=150,
        col_count=6
    ),
    visrag=VisRAGResult(
        recommendations=[
            VisRAGRecommendation(
                chart_family='line',
                rationale='Selected as a conservative fallback based on the request and available data shape.',
                priority=1,
                score=1.45,
                supporting_example_ids=[],
                supporting_corpora=[]
            )
        ],
        rules=[
            'Use clear titles and axis labels.',
            'Avoid overcrowded visuals.',
            'Prefer readable defaults.',
            'Prefer chart families supported by both the data profile and retrieved reference examples.'
        ],
        caveats

In [31]:
from langchain_ollama import ChatOllama

LLM_MODEL = "gemma3:1b"  # или "llama3.2:1b"
# LLM_MODEL = "llama3.2:1b"       # или "llama3.2:1b"
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,
)

In [32]:
from src.infrastructure import RuntimeContext

runtime = RuntimeContext(settings=settings, llm=llm)

# QueryUnderstandingService

In [33]:
from src.services import QueryUnderstandingService

qu = QueryUnderstandingService().invoke(runtime=runtime, user_context=request.user_context, query=request.query)

In [34]:
rprint("query:", request.query)
rprint(qu)

query: Show the sales trend over time

QueryUnderstandingResult(
    intent='Visualize sales trend over time',
    requested_operations=['Time series visualization', 'Trend analysis'],
    candidate_charts=['Line chart', 'Bar chart (for comparison)'],
    constraints=['Time period: Over what time range should the trend be visualized?'],
    case_type=<ChartCaseType.NON_CANONICAL: 'non_canonical'>,
    confidence=0.95
)

# CanonicalPlanningService

In [35]:
from src.services import CanonicalPlanningService

cp = CanonicalPlanningService().invoke(runtime=runtime, query_understanding=qu)

In [36]:
rprint(cp)

PlanningResult(
    mode=<ChartCaseType.CANONICAL: 'canonical'>,
    steps=[
        PlanningStep(
            name='1._define_time_period:_select_a',
            description='1. Define Time Period: Select a time range (e.g., last 30 days).'
        ),
        PlanningStep(
            name='2._data_preparation:_ensure_sales_data',
            description='2. Data Preparation: Ensure sales data is accessible and in a suitable format (e.g., CSV, 
database).'
        ),
        PlanningStep(
            name='3._line_chart_creation:_create_a',
            description="3. Line Chart Creation: Create a line chart with 'Date' as the x-axis and 'Sales' as the 
y-axis."
        ),
        PlanningStep(
            name='4._trend_identification:_analyze_the_line',
            description='4. Trend Identification: Analyze the line chart to identify the overall trend (increasing,
decreasing, stable).'
        ),
        PlanningStep(
            name='5._visualization_confirmation:_verify_the_identified',
            description='5. Visualization Confirmation: Verify the identified trend is visually consistent with the
data.'
        )
    ],
    success_criteria=[
        'At least one valid canonical chart is produced, preferably among: Line chart, Bar chart (for 
comparison).',
        'Generated charts are readable and consistent with the request.',
        'Final statements reference execution metrics or chart evidence.'
    ]
)

# NonCanonicalPlanningService

In [37]:
from src.services import NonCanonicalPlanningService

ncp = NonCanonicalPlanningService().invoke(runtime=runtime, query_understanding=qu)

In [38]:
rprint(ncp)

PlanningResult(
    mode=<ChartCaseType.NON_CANONICAL: 'non_canonical'>,
    steps=[
        PlanningStep(
            name='profile_the_dataset_and_identify_fields',
            description='Profile the dataset and identify fields that may support the requested non-canonical 
output.'
        ),
        PlanningStep(
            name='prepare_a_cleaned_and_constrained_working',
            description='Prepare a cleaned and constrained working dataset to reduce ambiguity.'
        ),
        PlanningStep(
            name='retrieve_broader_charting_guidance,_including_caveats',
            description='Retrieve broader charting guidance, including caveats and anti-patterns, for the requested
scenario.'
        ),
        PlanningStep(
            name='evaluate_whether_a_non_canonical_design',
            description='Evaluate whether a non-canonical design is necessary or whether simpler chart families 
such as Line chart, Bar chart (for comparison) can satisfy the request.'
        ),
        PlanningStep(
            name='generate_and_execute_an_initial_chart',
            description='Generate and execute an initial chart solution together with numeric summaries.'
        ),
        PlanningStep(
            name='read_chart_structure,_extract_evidence_backed',
            description='Read chart structure, extract evidence-backed facts and compare them against the intended 
message.'
        ),
        PlanningStep(
            name='run_an_explicit_verification_pass_to',
            description='Run an explicit verification pass to flag unsupported statements and unresolved 
assumptions.'
        )
    ],
    success_criteria=[
        'Cautious',
        'Conservative',
        'Strong Evidence Collection',
        'Assumption Checks',
        'Fallback Handling',
        'Verification'
    ]
)

# DataProfilerService

In [39]:
from src.services import DataProfilerService

data_profile = DataProfilerService().invoke(runtime=runtime, data_path=data_path)

In [40]:
rprint(data_profile)

DataProfile(
    row_count=150,
    col_count=6,
    columns=[
        DataColumnProfile(name='Id', dtype='numeric', missing_ratio=0.0, unique_count=150),
        DataColumnProfile(name='SepalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=35),
        DataColumnProfile(name='SepalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=23),
        DataColumnProfile(name='PetalLengthCm', dtype='numeric', missing_ratio=0.0, unique_count=43),
        DataColumnProfile(name='PetalWidthCm', dtype='numeric', missing_ratio=0.0, unique_count=22),
        DataColumnProfile(name='Species', dtype='categorical', missing_ratio=0.0, unique_count=3)
    ],
    likely_numeric_columns=['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm'],
    likely_categorical_columns=['Species'],
    likely_time_columns=[],
    quality_notes=["Column 'Id' looks like an identifier."]
)

# DataPreparationService

In [41]:
from src.services import DataPreparationService

data_prep = DataPreparationService().invoke(runtime=runtime, data_path=data_path, data_profile=data_profile, run_id="1")

In [42]:
rprint(data_prep)

DataPreparationResult(
    output_path='../demo_data/tmp_folder/artifacts/1/cleaned_data.csv',
    operations=[],
    row_count=150,
    col_count=6
)

# VisRAG

In [43]:
from src.services import VisRAGService

recommendations = VisRAGService().invoke(runtime=runtime, data_profile=data_profile, query_understanding=qu)

In [44]:
rprint(recommendations)

VisRAGResult(
    recommendations=[
        VisRAGRecommendation(
            chart_family='line chart',
            rationale='Selected as a conservative fallback based on the request and available data shape.',
            priority=1,
            score=1.4,
            supporting_example_ids=[],
            supporting_corpora=[]
        ),
        VisRAGRecommendation(
            chart_family='bar chart (for comparison)',
            rationale='Selected as a conservative fallback based on the request and available data shape.',
            priority=2,
            score=1.2,
            supporting_example_ids=[],
            supporting_corpora=[]
        )
    ],
    rules=[
        'Use clear titles and axis labels.',
        'Avoid overcrowded visuals.',
        'Prefer readable defaults.',
        'Prefer chart families supported by both the data profile and retrieved reference examples.',
        'Validate whether a simpler canonical chart can communicate the same message.'
    ],
    caveats=[
        'Time period: Over what time range should the trend be visualized?',
        'Non-canonical case requires conservative interpretation.'
    ],
    retrieved_examples=[],
    corpus_status=['No VisRAG corpus root configured; using heuristic-only recommendations.'],
    retrieval_strategy='hybrid_rule_retrieval'
)